# 📋 Kiểm Tra Quy Tắc Lọc Trạm (R1-R6) — Toàn Bộ 32 Trạm

Notebook này kiểm tra **từng Rule** trên **tất cả 32 trạm** và đưa ra kết quả **PASS / FAIL** rõ ràng.

| Rule | Tên | Ngưỡng |
|:---:|:---|:---|
| R1 | Data Integrity | Missing < 20% |
| R2 | Frozen Sensor | Frozen events < 50 (PM2.5) |
| R3 | Spatial Clustering | ≥1 hàng xóm trong 100km |
| R4 | Data Uniqueness | r < 0.99 với đa số trạm cùng cụm |
| R5 | Climate Homogeneity | Cùng vùng khí hậu trong cụm |
| R6 | Variance | Cụm có đa dạng loại trạm |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# --- Load info ---
info_df = pd.read_csv('data/info.csv')
print(info_df.head())

# --- Station metadata ---
station_meta = {
    1:  {'province': 'Hà Nội',     'district': 'Cầu Giấy',     'lat': 21.0323, 'lon': 105.8007, 'region': 'Bắc'},
    2:  {'province': 'Hà Nội',     'district': 'Thanh Xuân',    'lat': 20.9947, 'lon': 105.7998, 'region': 'Bắc'},
    3:  {'province': 'Hà Nội',     'district': 'Tây Hồ',       'lat': 21.0690, 'lon': 105.8105, 'region': 'Bắc'},
    4:  {'province': 'Thanh Hóa',  'district': 'Thọ Xuân',     'lat': 19.9500, 'lon': 105.5000, 'region': 'Bắc'},
    5:  {'province': 'Hà Nội',     'district': 'Gia Lâm',      'lat': 21.0535, 'lon': 106.0071, 'region': 'Bắc'},
    6:  {'province': 'Hà Nội',     'district': 'Ba Vì',        'lat': 21.0833, 'lon': 105.3833, 'region': 'Bắc'},
    7:  {'province': 'TP.HCM',     'district': 'Quận 1',       'lat': 10.7807, 'lon': 106.6994, 'region': 'Nam'},
    8:  {'province': 'TP.HCM',     'district': 'Quận 3',       'lat': 10.7749, 'lon': 106.6863, 'region': 'Nam'},
    9:  {'province': 'TP.HCM',     'district': 'Bình Thạnh',   'lat': 10.8033, 'lon': 106.6967, 'region': 'Nam'},
    10: {'province': 'TP.HCM',     'district': 'Quận 10',      'lat': 10.7682, 'lon': 106.6663, 'region': 'Nam'},
    11: {'province': 'TP.HCM',     'district': 'Quận 11',      'lat': 10.7638, 'lon': 106.6436, 'region': 'Nam'},
    12: {'province': 'TP.HCM',     'district': 'Bình Chánh',   'lat': 10.6954, 'lon': 106.5913, 'region': 'Nam'},
    13: {'province': 'Đà Nẵng',    'district': 'Thanh Khê',    'lat': 16.0706, 'lon': 108.1910, 'region': 'Trung'},
    14: {'province': 'Đà Nẵng',    'district': 'Sơn Trà',      'lat': 16.0607, 'lon': 108.2326, 'region': 'Trung'},
    15: {'province': 'Đà Nẵng',    'district': 'Liên Chiểu',   'lat': 16.0177, 'lon': 108.2038, 'region': 'Trung'},
    16: {'province': 'Hải Phòng',  'district': 'Đồ Sơn',       'lat': 20.7136, 'lon': 106.7894, 'region': 'Bắc'},
    17: {'province': 'Hải Phòng',  'district': 'Hồng Bàng',    'lat': 20.8607, 'lon': 106.6790, 'region': 'Bắc'},
    18: {'province': 'Cần Thơ',    'district': 'Ninh Kiều',    'lat': 10.0371, 'lon': 105.7883, 'region': 'Nam'},
    19: {'province': 'Cần Thơ',    'district': 'Cái Răng',     'lat': 10.0009, 'lon': 105.7510, 'region': 'Nam'},
    20: {'province': 'Cần Thơ',    'district': 'Bình Thủy',    'lat': 10.0743, 'lon': 105.7397, 'region': 'Nam'},
    21: {'province': 'Khánh Hòa',  'district': 'Cam Ranh',     'lat': 11.9214, 'lon': 109.1591, 'region': 'Trung'},
    22: {'province': 'Khánh Hòa',  'district': 'Nha Trang',    'lat': 12.2451, 'lon': 109.1943, 'region': 'Trung'},
    23: {'province': 'Khánh Hòa',  'district': 'Diên Khánh',   'lat': 12.2549, 'lon': 109.0933, 'region': 'Trung'},
    24: {'province': 'Đồng Nai',   'district': 'Biên Hòa',     'lat': 10.9447, 'lon': 106.8243, 'region': 'Nam'},
    25: {'province': 'Đồng Nai',   'district': 'Long Thành',   'lat': 10.7891, 'lon': 106.9503, 'region': 'Nam'},
    26: {'province': 'Đồng Nai',   'district': 'Nhơn Trạch',   'lat': 10.7229, 'lon': 106.8834, 'region': 'Nam'},
    27: {'province': 'Nam Định',   'district': 'TP. Nam Định',  'lat': 20.4339, 'lon': 106.1773, 'region': 'Bắc'},
    28: {'province': 'Thanh Hóa',  'district': 'TP. Thanh Hóa', 'lat': 19.8000, 'lon': 105.7667, 'region': 'Bắc'},
    29: {'province': 'Nghệ An',    'district': 'TP. Vinh',      'lat': 18.6734, 'lon': 105.6923, 'region': 'Trung'},
    30: {'province': 'Vĩnh Long',  'district': 'TP. Vĩnh Long', 'lat': 10.2537, 'lon': 105.9722, 'region': 'Nam'},
    31: {'province': 'Trà Vinh',   'district': 'TP. Trà Vinh',  'lat': 9.9472,  'lon': 106.3423, 'region': 'Nam'},
    32: {'province': 'An Giang',   'district': 'Rạch Giá',      'lat': 10.0124, 'lon': 105.0809, 'region': 'Nam'},
}

# --- Load ALL raw data ---
AIR_FEATURES = ['aqi', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2']
all_air = {}
for sid in range(1, 33):
    try:
        df = pd.read_csv(f'data/raw/air/air_{sid}.csv')
        df['datetime'] = pd.to_datetime(df['timestamp_local'])
        df = df.set_index('datetime').sort_index()
        all_air[sid] = df[AIR_FEATURES].copy()
    except:
        pass

print(f"Loaded {len(all_air)} stations")


FileNotFoundError: [Errno 2] No such file or directory: 'data/info.csv'

## R1: Data Integrity — Missing < 20%
Kiểm tra tỷ lệ dữ liệu bị khuyết trên từng feature chính (PM2.5) cho mỗi trạm.


In [ ]:
# === R1: DATA INTEGRITY ===
r1_results = {}
for sid in range(1, 33):
    if sid in all_air:
        missing_pct = all_air[sid]['pm25'].isna().mean() * 100
        avg_missing = all_air[sid].isna().mean().mean() * 100
        r1_results[sid] = {
            'PM2.5 Missing (%)': round(missing_pct, 2),
            'Avg All Features Missing (%)': round(avg_missing, 2),
            'R1 Pass': missing_pct < 20
        }

r1_df = pd.DataFrame(r1_results).T
r1_df.index.name = 'Station'

# Visualization
fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#2ecc71' if v else '#e74c3c' for v in r1_df['R1 Pass']]
bars = ax.barh(range(len(r1_df)), r1_df['PM2.5 Missing (%)'], color=colors)
ax.axvline(x=20, color='red', linestyle='--', linewidth=2, label='Ngưỡng 20%')
ax.set_yticks(range(len(r1_df)))
ax.set_yticklabels([f"Trạm {int(s)}" for s in r1_df.index])
ax.set_xlabel('PM2.5 Missing (%)')
ax.set_title('R1: Data Integrity — Missing Values (🟢 PASS | 🔴 FAIL)', fontsize=14, fontweight='bold')
ax.legend()
for i, (v, p) in enumerate(zip(r1_df['PM2.5 Missing (%)'], r1_df['R1 Pass'])):
    ax.text(v + 0.3, i, f"{v:.1f}% {'✅' if p else '❌'}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nR1 PASS: {r1_df['R1 Pass'].sum()}/32 trạm")
print(f"R1 FAIL: {(~r1_df['R1 Pass']).sum()}/32 trạm")


## R2: Frozen Sensor — Frozen Events < 50 (PM2.5)
Đếm số lần PM2.5 lặp lại cùng giá trị ≥ 12 giờ liên tiếp. Nếu > 50 lần → cảm biến có vấn đề.


In [ ]:
# === R2: FROZEN SENSOR ===
def count_frozen(series, min_hours=12):
    if series.isna().all():
        return 0
    same = (series == series.shift()).astype(int)
    groups = same.ne(same.shift()).cumsum()
    frozen_counts = same.groupby(groups).sum()
    return (frozen_counts >= min_hours).sum()

r2_results = {}
for sid in range(1, 33):
    if sid in all_air:
        frozen_pm25 = count_frozen(all_air[sid]['pm25'])
        frozen_so2 = count_frozen(all_air[sid]['so2'])
        r2_results[sid] = {
            'Frozen PM2.5': frozen_pm25,
            'Frozen SO2': frozen_so2,
            'R2 Pass': frozen_pm25 < 50
        }

r2_df = pd.DataFrame(r2_results).T
r2_df.index.name = 'Station'

fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#2ecc71' if v else '#e74c3c' for v in r2_df['R2 Pass']]
ax.barh(range(len(r2_df)), r2_df['Frozen PM2.5'], color=colors)
ax.axvline(x=50, color='red', linestyle='--', linewidth=2, label='Ngưỡng 50')
ax.set_yticks(range(len(r2_df)))
ax.set_yticklabels([f"Trạm {int(s)}" for s in r2_df.index])
ax.set_xlabel('Số sự kiện Frozen (PM2.5)')
ax.set_title('R2: Frozen Sensor — PM2.5 đóng băng ≥12h (🟢 PASS | 🔴 FAIL)', fontsize=14, fontweight='bold')
ax.legend()
for i, (v, p) in enumerate(zip(r2_df['Frozen PM2.5'], r2_df['R2 Pass'])):
    ax.text(v + 0.5, i, f"{int(v)} {'✅' if p else '❌'}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nR2 PASS: {r2_df['R2 Pass'].sum()}/32 trạm")
print(f"R2 FAIL: {(~r2_df['R2 Pass']).sum()}/32 trạm")
failed = r2_df[~r2_df['R2 Pass']]
if len(failed) > 0:
    print("\nTrạm FAIL R2:")
    for sid in failed.index:
        m = station_meta[int(sid)]
        print(f"  Trạm {int(sid)} ({m['district']}, {m['province']}): {int(failed.loc[sid, 'Frozen PM2.5'])} events")


## R3: Spatial Clustering — ≥1 hàng xóm trong 100km
Tính khoảng cách Haversine giữa tất cả cặp trạm. Trạm nào không có bất kỳ trạm nào khác trong bán kính 100km → "mồ côi" → FAIL.


In [ ]:
# === R3: SPATIAL CLUSTERING ===
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Build distance matrix
stations = list(range(1, 33))
dist_matrix = pd.DataFrame(index=stations, columns=stations, dtype=float)
for s1 in stations:
    for s2 in stations:
        m1, m2 = station_meta[s1], station_meta[s2]
        dist_matrix.loc[s1, s2] = haversine(m1['lat'], m1['lon'], m2['lat'], m2['lon'])

# R3 check: at least 1 neighbor within 100km
r3_results = {}
for sid in stations:
    others = dist_matrix.loc[sid].drop(sid)
    neighbors_100 = (others < 100).sum()
    nearest = others.min()
    nearest_id = others.idxmin()
    r3_results[sid] = {
        'Neighbors <100km': neighbors_100,
        'Nearest Station': nearest_id,
        'Nearest Distance (km)': round(nearest, 1),
        'R3 Pass': neighbors_100 >= 1
    }

r3_df = pd.DataFrame(r3_results).T
r3_df.index.name = 'Station'

fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#2ecc71' if v else '#e74c3c' for v in r3_df['R3 Pass']]
ax.barh(range(len(r3_df)), r3_df['Neighbors <100km'], color=colors)
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, label='Ngưỡng ≥1')
ax.set_yticks(range(len(r3_df)))
ax.set_yticklabels([f"Trạm {int(s)}" for s in r3_df.index])
ax.set_xlabel('Số hàng xóm trong 100km')
ax.set_title('R3: Spatial Clustering — Số hàng xóm <100km (🟢 PASS | 🔴 FAIL)', fontsize=14, fontweight='bold')
ax.legend()
for i, (v, p) in enumerate(zip(r3_df['Neighbors <100km'], r3_df['R3 Pass'])):
    ax.text(v + 0.2, i, f"{int(v)} {'✅' if p else '❌'}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

# Distance heatmap for visual check
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(dist_matrix.astype(float), annot=True, fmt='.0f', cmap='RdYlGn_r',
            ax=ax, linewidths=0.2, annot_kws={'size': 6}, vmin=0, vmax=500)
ax.set_title('Ma Trận Khoảng Cách Haversine (km) — 32 Trạm', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nR3 PASS: {r3_df['R3 Pass'].sum()}/32 trạm")
failed = r3_df[~r3_df['R3 Pass']]
if len(failed) > 0:
    print("Trạm MỒ CÔI (FAIL R3):")
    for sid in failed.index:
        m = station_meta[int(sid)]
        print(f"  Trạm {int(sid)} ({m['district']}, {m['province']}): nearest = Trạm {int(failed.loc[sid, 'Nearest Station'])} ({failed.loc[sid, 'Nearest Distance (km)']}km)")


## R4: Data Uniqueness — r < 0.99 với đa số trạm
Tính correlation PM2.5 daily giữa tất cả cặp trạm. Trạm nào có r ≥ 0.99 với ≥3 trạm khác → data trùng lặp (cùng ô lưới Reanalysis) → FAIL.


In [ ]:
# === R4: DATA UNIQUENESS ===

# Build PM2.5 daily correlation matrix
pm25_all = pd.DataFrame()
for sid in range(1, 33):
    if sid in all_air and 'pm25' in all_air[sid].columns:
        pm25_all[sid] = all_air[sid]['pm25']

pm25_daily = pm25_all.resample('D').mean()
corr_matrix = pm25_daily.corr()

# R4 check: count how many stations have r >= 0.99
r4_results = {}
for sid in range(1, 33):
    if sid in corr_matrix.columns:
        corr_row = corr_matrix[sid].drop(sid)
        high_corr_count = (corr_row >= 0.99).sum()
        high_corr_stations = list(corr_row[corr_row >= 0.99].index)
        r4_results[sid] = {
            'Stations r≥0.99': high_corr_count,
            'Duplicate With': str(high_corr_stations) if high_corr_stations else 'None',
            'R4 Pass': high_corr_count < 3
        }

r4_df = pd.DataFrame(r4_results).T
r4_df.index.name = 'Station'

fig, ax = plt.subplots(figsize=(16, 8))
colors = ['#2ecc71' if v else '#e74c3c' for v in r4_df['R4 Pass']]
ax.barh(range(len(r4_df)), r4_df['Stations r≥0.99'].astype(int), color=colors)
ax.axvline(x=3, color='red', linestyle='--', linewidth=2, label='Ngưỡng <3')
ax.set_yticks(range(len(r4_df)))
ax.set_yticklabels([f"Trạm {int(s)}" for s in r4_df.index])
ax.set_xlabel('Số trạm có r ≥ 0.99')
ax.set_title('R4: Data Uniqueness — Số trạm trùng data r≥0.99 (🟢 PASS | 🔴 FAIL)', fontsize=14, fontweight='bold')
ax.legend()
for i, (v, p) in enumerate(zip(r4_df['Stations r≥0.99'], r4_df['R4 Pass'])):
    ax.text(int(v) + 0.2, i, f"{int(v)} {'✅' if p else '❌'}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nR4 PASS: {r4_df['R4 Pass'].sum()}/32 trạm")
failed = r4_df[~r4_df['R4 Pass']]
if len(failed) > 0:
    print("\nTrạm FAIL R4 (data trùng lặp):")
    for sid in failed.index:
        m = station_meta[int(sid)]
        print(f"  Trạm {int(sid)} ({m['district']}, {m['province']}): trùng với {failed.loc[sid, 'Duplicate With']}")


## R5: Climate Homogeneity — Phân vùng khí hậu
Chia 32 trạm thành 3 vùng khí hậu (Bắc / Trung / Nam) dựa trên vĩ độ và đặc trưng nhiệt độ. Trạm miền Trung cô lập sẽ FAIL vì không thể gộp vào cụm nào.

**Quy tắc phân vùng:**
- Bắc: Vĩ độ > 18° (Có mùa đông lạnh, invesion layer gây ô nhiễm cực đoan)
- Trung: 12° < Vĩ độ ≤ 18° (Chuyển tiếp, bão lũ, đèo Hải Vân chia cắt)
- Nam: Vĩ độ ≤ 12° (Nhiệt đới, không có mùa đông)


In [ ]:
# === R5: CLIMATE HOMOGENEITY ===

r5_results = {}
for sid in range(1, 33):
    m = station_meta[sid]
    lat = m['lat']
    region = m['region']
    
    # Trạm Trung Bộ: FAIL vì không gộp được vào cụm nào
    # (Bắc quá xa, Nam quá xa, tự lập cụm thì quá ít trạm)
    if region == 'Trung':
        r5_pass = False
        r5_note = f"Vùng Trung Bộ (Lat={lat:.1f}°) — Không gộp được vào Bắc hay Nam"
    else:
        r5_pass = True
        r5_note = f"Vùng {region} Bộ (Lat={lat:.1f}°) — Phù hợp cụm {'Bắc' if region == 'Bắc' else 'Nam'}"
    
    r5_results[sid] = {
        'Region': region,
        'Latitude': lat,
        'R5 Note': r5_note,
        'R5 Pass': r5_pass
    }

r5_df = pd.DataFrame(r5_results).T
r5_df.index.name = 'Station'

fig, ax = plt.subplots(figsize=(16, 8))
region_colors = {'Bắc': '#ff6666', 'Nam': '#6666ff', 'Trung': '#cccccc'}
colors = [region_colors[r] for r in r5_df['Region']]
ax.barh(range(len(r5_df)), r5_df['Latitude'].astype(float), color=colors)
ax.axvline(x=18, color='orange', linestyle='--', linewidth=2, label='Ranh Bắc/Trung (18°)')
ax.axvline(x=12, color='purple', linestyle='--', linewidth=2, label='Ranh Trung/Nam (12°)')
ax.set_yticks(range(len(r5_df)))
ax.set_yticklabels([f"Trạm {int(s)}" for s in r5_df.index])
ax.set_xlabel('Vĩ độ (°N)')
ax.set_title('R5: Climate Homogeneity — Phân vùng khí hậu (🔴Bắc 🔵Nam ⚪Trung=FAIL)', fontsize=14, fontweight='bold')
ax.legend()
for i, (v, p) in enumerate(zip(r5_df['Latitude'], r5_df['R5 Pass'])):
    ax.text(float(v) + 0.2, i, f"{'✅' if p else '❌'}", va='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nR5 PASS: {r5_df['R5 Pass'].sum()}/32 trạm")
print(f"R5 FAIL (Trung Bộ cô lập): {(~r5_df['R5 Pass']).sum()}/32 trạm")


## R6: Variance — Đa dạng loại trạm
Rule R6 áp dụng ở cấp **cụm** (cluster-level), không ở cấp từng trạm. Kiểm tra xem mỗi cụm đề xuất có đủ đa dạng: đô thị, ngoại ô, công nghiệp, biển/núi.


In [ ]:
# === R6: VARIANCE (cluster-level) ===

station_types = {
    1: 'Đô thị giao thông', 2: 'Đô thị giao thông', 3: 'Đô thị nền',
    4: 'Nền nông nghiệp', 5: 'Vành đai ven đô', 6: 'Ngoại ô vùng núi',
    7: 'Lõi đô thị', 8: 'Lõi đô thị', 9: 'Đô thị giao thông',
    10: 'Lõi đô thị', 11: 'Lõi đô thị', 12: 'Vành đai ven đô',
    13: 'Đô thị ven biển', 14: 'Ven biển', 15: 'Công nghiệp ven biển',
    16: 'Biển', 17: 'Công nghiệp/cảng', 18: 'Đô thị ĐBSCL',
    19: 'Đô thị ĐBSCL', 20: 'Đô thị ĐBSCL', 21: 'Ven biển',
    22: 'Ven biển', 23: 'Nền nông nghiệp', 24: 'Công nghiệp nặng',
    25: 'Công trường/sân bay', 26: 'KCN vệ tinh', 27: 'Đô thị vệ tinh',
    28: 'Đô thị chuyển tiếp', 29: 'Đô thị tỉnh lẻ', 30: 'Đô thị nhỏ ĐBSCL',
    31: 'Ven biển ĐBSCL', 32: 'Biên giới'
}

# Proposed clusters
cluster_north = [1, 2, 3, 4, 5, 6, 16, 17, 27, 28]
cluster_south = [7, 18, 24, 30, 31, 32]

print("=" * 60)
print("CỤM BẮC BỘ — Đa dạng loại trạm:")
print("=" * 60)
north_types = set()
for sid in cluster_north:
    t = station_types[sid]
    north_types.add(t)
    print(f"  Trạm {sid:2d} ({station_meta[sid]['district']:15s}): {t}")
print(f"\n  → Tổng loại trạm unique: {len(north_types)} ({'✅ R6 PASS' if len(north_types) >= 3 else '❌ R6 FAIL'})")
print(f"  → Các loại: {north_types}")

print(f"\n{'=' * 60}")
print("CỤM NAM BỘ — Đa dạng loại trạm:")
print("=" * 60)
south_types = set()
for sid in cluster_south:
    t = station_types[sid]
    south_types.add(t)
    print(f"  Trạm {sid:2d} ({station_meta[sid]['district']:15s}): {t}")
print(f"\n  → Tổng loại trạm unique: {len(south_types)} ({'✅ R6 PASS' if len(south_types) >= 3 else '❌ R6 FAIL'})")
print(f"  → Các loại: {south_types}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, (name, ids, color) in zip(axes, [('Bắc Bộ', cluster_north, '#ff6666'), ('Nam Bộ', cluster_south, '#6666ff')]):
    types = [station_types[s] for s in ids]
    type_counts = pd.Series(types).value_counts()
    type_counts.plot(kind='barh', color=color, ax=ax, edgecolor='black')
    ax.set_title(f'R6: Variance — Cụm {name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Số trạm')

plt.tight_layout()
plt.show()


## 📊 BẢNG TỔNG HỢP: Pass/Fail Tất Cả Rules — 32 Trạm


In [ ]:
# === FINAL SUMMARY TABLE ===

summary = []
for sid in range(1, 33):
    m = station_meta[sid]
    row = {
        'Station': sid,
        'Province': m['province'],
        'District': m['district'],
        'Region': m['region'],
        'R1': '✅' if r1_df.loc[sid, 'R1 Pass'] else '❌',
        'R2': '✅' if r2_df.loc[sid, 'R2 Pass'] else '❌',
        'R3': '✅' if r3_df.loc[sid, 'R3 Pass'] else '❌',
        'R4': '✅' if r4_df.loc[sid, 'R4 Pass'] else '❌',
        'R5': '✅' if r5_df.loc[sid, 'R5 Pass'] else '❌',
    }
    
    # R6 is cluster-level
    if sid in cluster_north or sid in cluster_south:
        row['R6'] = '✅'
    else:
        row['R6'] = '—'
    
    all_pass = all([
        r1_df.loc[sid, 'R1 Pass'],
        r2_df.loc[sid, 'R2 Pass'],
        r3_df.loc[sid, 'R3 Pass'],
        r4_df.loc[sid, 'R4 Pass'],
        r5_df.loc[sid, 'R5 Pass'],
    ])
    
    if sid in cluster_north:
        row['Proposed'] = '🔴 Bắc'
    elif sid in cluster_south:
        row['Proposed'] = '🔵 Nam'
    else:
        row['Proposed'] = '❌ Loại'
    
    row['All Pass'] = '✅' if all_pass else '❌'
    summary.append(row)

summary_df = pd.DataFrame(summary).set_index('Station')
display(summary_df)

# Count
print(f"\n{'='*50}")
print(f"TỔNG KẾT:")
print(f"  Trạm PASS tất cả R1-R5: {(summary_df['All Pass'] == '✅').sum()}")
print(f"  Trạm FAIL ít nhất 1 rule: {(summary_df['All Pass'] == '❌').sum()}")
print(f"\n  🔴 Cụm Bắc Bộ: {len(cluster_north)} trạm — {cluster_north}")
print(f"  🔵 Cụm Nam Bộ: {len(cluster_south)} trạm — {cluster_south}")
print(f"  Tổng được chọn: {len(cluster_north) + len(cluster_south)} trạm")
print(f"\nSELECTED_STATIONS = {sorted(cluster_north + cluster_south)}")
